In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.datasets.brats_dataset import create_file_list
from src.config import DATA_DIR
from src.transforms.preprocessing import (
    train_transform,
    val_transform
)

In [3]:
files = create_file_list(DATA_DIR)

print(len(files))

369


In [4]:
from sklearn.model_selection import train_test_split

files = create_file_list(DATA_DIR)

train_files, val_files = train_test_split(
    files,
    test_size=0.2,
    random_state=42
)

print(len(train_files))
print(len(val_files))

295
74


In [5]:
from monai.transforms import Compose, LoadImaged

debug_transform = Compose([
    LoadImaged(keys=["image", "label"])
])

sample = debug_transform(files[0])

print("Image shape:", sample["image"].shape)
print("Label shape:", sample["label"].shape)

Image shape: torch.Size([4, 240, 240, 155])
Label shape: torch.Size([240, 240, 155])


In [6]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd
)

debug_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"])
])

sample = debug_transform(files[0])

print("Image shape:", sample["image"].shape)
print("Label shape:", sample["label"].shape)

Image shape: torch.Size([4, 240, 240, 155])
Label shape: torch.Size([1, 240, 240, 155])


In [7]:
print(files[0]["image"])
print(files[0]["label"])

['D:\\Internship\\brats2020\\Data\\BraTS2020_TrainingData\\MICCAI_BraTS2020_TrainingData\\BraTS20_Training_001\\BraTS20_Training_001_flair.nii', 'D:\\Internship\\brats2020\\Data\\BraTS2020_TrainingData\\MICCAI_BraTS2020_TrainingData\\BraTS20_Training_001\\BraTS20_Training_001_t1.nii', 'D:\\Internship\\brats2020\\Data\\BraTS2020_TrainingData\\MICCAI_BraTS2020_TrainingData\\BraTS20_Training_001\\BraTS20_Training_001_t1ce.nii', 'D:\\Internship\\brats2020\\Data\\BraTS2020_TrainingData\\MICCAI_BraTS2020_TrainingData\\BraTS20_Training_001\\BraTS20_Training_001_t2.nii']
D:\Internship\brats2020\Data\BraTS2020_TrainingData\MICCAI_BraTS2020_TrainingData\BraTS20_Training_001\BraTS20_Training_001_seg.nii


In [8]:
from monai.data import Dataset

train_ds = Dataset(
    data=train_files,
    transform=train_transform
)

val_ds = Dataset(
    data=val_files,
    transform=val_transform
)

In [9]:
from src.graphs.create_graph_dataset import create_graph_dataset

In [10]:
train_graphs = create_graph_dataset(train_ds)

val_graphs = create_graph_dataset(val_ds)

  0%|          | 0/295 [00:00<?, ?it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  0%|          | 1/295 [00:00<04:43,  1.04it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 2/295 [00:01<04:28,  1.09it/s]d:\Internship\brats2020\src\graphs\graph_dataset.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(
  1%|          | 3/295 [

In [29]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)

In [ ]:
from train.train_gcn import train_gcn

model = train_gcn(
    train_loader,
    val_loader,
    epochs=200,
    lr=5e-4
)

Epoch 000 | Train Loss: 0.4203 | Train Acc: 0.9101 | Val Loss: 0.1095 | Val Acc: 0.9892
Epoch 010 | Train Loss: 0.0755 | Train Acc: 0.9731 | Val Loss: 0.0239 | Val Acc: 0.9915
Epoch 020 | Train Loss: 0.0689 | Train Acc: 0.9748 | Val Loss: 0.0228 | Val Acc: 0.9920
Epoch 030 | Train Loss: 0.0673 | Train Acc: 0.9753 | Val Loss: 0.0223 | Val Acc: 0.9919
Epoch 040 | Train Loss: 0.0655 | Train Acc: 0.9758 | Val Loss: 0.0219 | Val Acc: 0.9922
Epoch 050 | Train Loss: 0.0638 | Train Acc: 0.9759 | Val Loss: 0.0225 | Val Acc: 0.9920
Epoch 060 | Train Loss: 0.0650 | Train Acc: 0.9760 | Val Loss: 0.0226 | Val Acc: 0.9921
Epoch 070 | Train Loss: 0.0620 | Train Acc: 0.9767 | Val Loss: 0.0234 | Val Acc: 0.9919
Epoch 080 | Train Loss: 0.0614 | Train Acc: 0.9773 | Val Loss: 0.0208 | Val Acc: 0.9926
Epoch 090 | Train Loss: 0.0605 | Train Acc: 0.9772 | Val Loss: 0.0214 | Val Acc: 0.9925
Epoch 100 | Train Loss: 0.0588 | Train Acc: 0.9779 | Val Loss: 0.0222 | Val Acc: 0.9922
Epoch 110 | Train Loss: 0.0577 |

In [31]:
print(train_transform)

In [35]:
from sklearn.metrics import classification_report, confusion_matrix
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.eval()

all_true = []
all_pred = []

with torch.no_grad():

    for graph in val_loader:

        graph = graph.to(device)

        out = model(
            graph.x,
            graph.edge_index
        )

        pred = out.argmax(dim=1)

        all_true.extend(
            graph.y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")
print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3]
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    129029
           1       0.69      0.39      0.50       417
           2       0.61      0.57      0.59       867
           3       0.58      0.30      0.40       223

    accuracy                           0.99    130536
   macro avg       0.72      0.57      0.62    130536
weighted avg       0.99      0.99      0.99    130536


Confusion Matrix:

[[128870     32    112     15]
 [    98    163    129     27]
 [   334     28    497      8]
 [    70     12     73     68]]


In [34]:
import numpy as np


def dice_score(y_true, y_pred, cls):

    y_true = (y_true == cls)
    y_pred = (y_pred == cls)

    intersection = np.sum(
        y_true & y_pred
    )

    return (
        2 * intersection
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + 1e-8
    )


print("\n===== Dice Scores =====")

for cls in range(4):

    dice = dice_score(
        np.array(all_true),
        np.array(all_pred),
        cls
    )

    print(
        f"Class {cls}: {dice:.4f}"
    )


===== Dice Scores =====
Class 0: 0.9974
Class 1: 0.5000
Class 2: 0.5924
Class 3: 0.3988
